In [65]:
# 08/2025
# Load distance matrices, generate isomap/Umap
# Join the above generated shape info with Database-specific infomation
#     add function_info to make complete_info
# Verifications
# Save the csv file if needed

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
curProject = 'HD'
curRegion = 'CST'  # CSSyl, CSpreCS, CST
curRoot = 'C'  # 'C' or 'D'

In [9]:
#######################################  load distances  #####################################

##  reading from the basic analysis  ##
#distance_path_min = rf'{curRoot}:\B_projWIP\proj_{curProject}\WINHD_MOTOHD_HDTRACK\{curRegion}\Isomap\minDist{curRegion}.txt'
#distance_path_max = rf'{curRoot}:\B_projWIP\proj_{curProject}\WINHD_MOTOHD_HDTRACK\{curRegion}\Isomap\maxDist{curRegion}.txt'
distance_path_min = rf'{curRoot}:\B_projWIP\proj_{curProject}\MOTOHD_Pyramidal\linux_results\minDist{curRegion}.txt'
distance_path_max = rf'{curRoot}:\B_projWIP\proj_{curProject}\MOTOHD_Pyramidal\linux_results\maxDist{curRegion}.txt'

try:
    minDist = pd.read_csv(distance_path_min, index_col=0, header=0)
    maxDist = pd.read_csv(distance_path_max, index_col=0, header=0)    
except FileNotFoundError as e:
    print(f"Error: {e}")

rows, cols = minDist.shape
print(f"Number of rows: {rows}")
print(f"Number of columns: {cols}")

Number of rows: 42
Number of columns: 42


In [13]:
################################    generation of Isomap    ##################################
# generation of isomaps using minDist and maxDist, all subjects, NO selection
# Define: outNameMin/Max, outFileNameMin/Max

from sklearn.manifold import Isomap
numDim = 6
numNeig = 30       #test: 10, 30, 60      MODIFY if needed
genIsomap = False  ## !!!!!!!!!!!!!  default to False, True only after varification  !!!!!!!!!!!!!

outNameMin = 'isomapCmds'+curRegion+'k'+str(numNeig)+'d'+str(numDim)+'distmin'+'.txt'
outNameMax = 'isomapCmds'+curRegion+'k'+str(numNeig)+'d'+str(numDim)+'distmax'+'.txt'

outFileNameMin = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\form_measure\isomap\{outNameMin}'
outFileNameMax = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\form_measure\isomap\{outNameMax}'

subjNames = minDist.index
dimNames = np.arange(1,numDim+1)
iso_min = Isomap(n_neighbors=numNeig,n_components=numDim,metric='precomputed').fit_transform(minDist.values)
iso_min_DF = pd.DataFrame(iso_min,index=subjNames,columns=dimNames)
iso_max = Isomap(n_neighbors=numNeig,n_components=numDim,metric='precomputed').fit_transform(maxDist.values)
iso_max_DF = pd.DataFrame(iso_max,index=subjNames,columns=dimNames)

# SAVE Isomaps as csv
print(outFileNameMin)
print(outFileNameMax)
if genIsomap:
    iso_min_DF.to_csv(outFileNameMin,index_label='subjName')
    iso_max_DF.to_csv(outFileNameMax,index_label='subjName')

C:\B_projWIP\proj_HD\Analysis_2025\form_measure\isomap\isomapCmdsCSTk30d6distmin.txt
C:\B_projWIP\proj_HD\Analysis_2025\form_measure\isomap\isomapCmdsCSTk30d6distmax.txt


In [31]:
###############################   generation of UMAP   ################################
# UMAP for the original distance WITHOUT selection of subjects
import umap
import random

# to ensure that the results are always the same
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

writeUMAP = False     # default to False, True only after varification            ###   CHANGE   ###

typeDist = 'min'   # using minimum or maximum distance matrix                     ###   CHANGE   ###
n_comp = 1         # Change this to the desired number of dimensions: 1 or 2      ###   CHANGE   ###
n_neighbors = 5    # Change this to the desired number of neighbors: 5 or 30      ###   CHANGE   ###
min_dist = 0.2     # Change this to the desired minimum distance
# define df for UMAP generation
if (typeDist == 'min'):
    df = minDist    
if (typeDist == 'max'):
    df = maxDist        
    
###############################  define output file name  ############################
outName = 'dim'+str(n_comp)+'_'+typeDist+'_neig'+str(n_neighbors)+'_dist'+str(min_dist)+'.txt'
###############################  define output file path  ############################   

curTypeAnalysis = 'main_piece_analysis' # 'basic_analysis'  ##  !!!!!!!!   CHANGE  !!!!!!!!
outFileName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\form_measure\umap_{curRegion}\{outName}'
##############################################################################################################################
# perform UMAP
reducer = umap.UMAP(metric='precomputed', n_components=n_comp,n_neighbors=n_neighbors,min_dist=min_dist,random_state=SEED)
embedding = reducer.fit_transform(df)

# Create a DataFrame for the embedding
if n_comp == 1:
    embedding_df = pd.DataFrame(embedding, columns=['UMAP1']) 
if n_comp == 2:
    embedding_df = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2'])
if n_comp == 3:
    embedding_df = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2', 'UMAP3'])
embedding_df.index = df.index

print(embedding_df)
print(outFileName)
# Save as csv
if writeUMAP:
    embedding_df.to_csv(outFileName,index_label='subjName')

C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
C:\Users\joyca\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


                    UMAP1      UMAP2
subjName                            
L010017MHU      -3.005078  16.147840
L010023MHU      -2.853867  18.024328
L010052MHU      -3.381294  18.122494
L010086MHU      -2.718785  14.973421
L010172MHU      -1.865253  17.351866
L010174MHU      -2.550139  17.671791
L010179MHU      -1.676720  14.302656
L010210MHU      -2.771921  14.089902
L010218MHU      -3.738398  15.699811
L010264MHU      -1.690776  14.948927
L010290MHU      -2.788072  15.587862
L010339MHU      -1.365095  14.612383
L010408MHU      -3.147432  17.278307
L010540MHU      -0.928725  14.415215
L010606MHU      -1.119816  15.447371
L010697MHU      -1.446814  16.597219
L010820MHU      -1.969097  16.904852
L011006MHU      -1.862759  15.574245
L011042MHU      -3.187427  16.720587
L011119MHU      -3.561539  18.528194
L011160MHU      -2.353181  14.565255
flip-R010017MHU -2.700279  16.388964
flip-R010023MHU -3.031338  18.723066
flip-R010052MHU -3.219377  17.741243
flip-R010086MHU -3.066823  15.196075
f

In [81]:
##########################  Load shape measures, if generated and written above  #############################
curDistType = 'min'                                         ##############   CHANGE  ###############
curNeig = '30'  # 10, 30, 60                                ##############   CHANGE  ###############
curTypeAnalysis = 'form_measure'                            ##############   CHANGE  ###############
#curTypeAnalysis = 'main_piece_analysis' 

#shape_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\isomap\isomapCmds{curRegion}k{curNeig}d3dist{curDistType}.txt'
# more dimensions of isomap
shape_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\isomap\isomapCmds{curRegion}k{curNeig}d6dist{curDistType}.txt'
shapeU_1_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\umap_{curRegion}\dim1_{curDistType}_neig5_dist0.2.txt'
shapeU_2_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\umap_{curRegion}\dim1_{curDistType}_neig30_dist0.2.txt'
shapeU_3_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\umap_{curRegion}\dim2_{curDistType}_neig5_dist0.2.txt'
shapeU_4_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curTypeAnalysis}\umap_{curRegion}\dim2_{curDistType}_neig30_dist0.2.txt'

print(shape_path)
print(shapeU_1_path)
print(shapeU_2_path)
print(shapeU_3_path)
print(shapeU_4_path)

C:\B_projWIP\proj_HD\Analysis_2025\form_measure\isomap\isomapCmdsCSTk30d6distmin.txt
C:\B_projWIP\proj_HD\Analysis_2025\form_measure\umap_CST\dim1_min_neig5_dist0.2.txt
C:\B_projWIP\proj_HD\Analysis_2025\form_measure\umap_CST\dim1_min_neig30_dist0.2.txt
C:\B_projWIP\proj_HD\Analysis_2025\form_measure\umap_CST\dim2_min_neig5_dist0.2.txt
C:\B_projWIP\proj_HD\Analysis_2025\form_measure\umap_CST\dim2_min_neig30_dist0.2.txt


In [83]:
#######################  Load shape information  #######################
shape = shapeU_1 = shapeU_2 = shapeU_3 = shapeU_4 = None
try:
    shape = pd.read_csv(shape_path, index_col=0, header=0)
    shapeU_1 = pd.read_csv(shapeU_1_path, index_col=0, header=0)
    shapeU_2 = pd.read_csv(shapeU_2_path, index_col=0, header=0)
    shapeU_3 = pd.read_csv(shapeU_3_path, index_col=0, header=0)
    shapeU_4 = pd.read_csv(shapeU_4_path, index_col=0, header=0)

    # Add 'iso' to column number, for any number of dimensions (1, 3, 10, etc.)
    shape.rename(columns=lambda x: f"iso{x}" if x.isdigit() else x, inplace=True)
    # renmae umap columns
    shapeU_1.rename(columns={'UMAP1': 'UMAP1_U1'}, inplace=True)
    shapeU_2.rename(columns={'UMAP1': 'UMAP1_U2'}, inplace=True)
    shapeU_3.rename(columns={'UMAP1': 'UMAP1_U3', 'UMAP2': 'UMAP2_U3'}, inplace=True)
    shapeU_4.rename(columns={'UMAP1': 'UMAP1_U4', 'UMAP2': 'UMAP2_U4'}, inplace=True)
    #shapeU_4.rename(columns={'UMAP1': 'UMAP1_U4', 'UMAP2': 'UMAP2_U4','UMAP3': 'UMAP3_U4'}, inplace=True)
except FileNotFoundError as e:
    print(f"Error: {e}")

if all(df is not None for df in [shape, shapeU_1, shapeU_2, shapeU_3, shapeU_4]):
    shape_joined = shape.join([shapeU_1, shapeU_2, shapeU_3, shapeU_4])
    shape = shape_joined
    print(shape.head())
    print(shape.index)
else:
    print("One or more input files were missing — join was not performed.")
    

                iso1      iso2      iso3      iso4      iso5      iso6  \
subjName                                                                 
L010017MHU -1.779720 -0.667942 -0.097973  1.451221 -0.281212  0.210212   
L010023MHU -0.281341  0.077161  0.301623 -0.250930 -0.780885  1.818217   
L010052MHU -0.784505 -0.755314  0.740993 -0.692597 -0.719726 -0.477771   
L010086MHU  0.095304  0.313703 -0.254081  0.338678 -0.089995  0.175775   
L010172MHU  2.021033  3.362425 -0.214150  0.552886 -1.296802 -0.497184   

             UMAP1_U1   UMAP1_U2  UMAP1_U3  UMAP2_U3  UMAP1_U4  UMAP2_U4  
subjName                                                                  
L010017MHU  18.420740  19.899364  1.210756 -0.001990 -0.796567  3.202875  
L010023MHU  19.885485  17.047668  4.303838  1.361453 -2.235939  1.729583  
L010052MHU  19.193771  20.170938  2.249991 -1.108080 -1.084145  2.748239  
L010086MHU  15.953506  16.603138  3.669662  1.601061 -1.329253  1.533374  
L010172MHU  13.810039  16.17897

In [85]:
###############################  Proecssing the shape file  ################################
# Add a 'SubjID' column, based on index, removing L, filp_R as prefix
# Add a 'Hemisphere' column (Left if index starts with L or Right if index starts with flip)

# Create SubjID from index, removing pre_fix and post_fix
shape['SubjID'] = (
    shape.index
    .to_series()  # convert index to a Series so we can use string operations
    .astype(str)   # convert index values to strings
    .str.replace(r'^(L|flip-R)', '', regex=True)   # remove 'L' or 'flip-R' at start
)
shape['SubjID'] = shape['SubjID'].astype(str) # Make sure SubjID is string
shape['Study'] = 'MOTOHD' # Default Study = MOTOHD
shape.loc[shape['SubjID'].str.startswith('v'), 'Study'] = 'HDTRACK' # Assign HDTRACT where SubjID starts with 'v'
shape.loc[shape['SubjID'].str.startswith('W'), 'Study'] = 'WINHD' # Assign HDTRACT where SubjID starts with 'v'

# Create 'Hemisphere' based on index prefix
#shape['Hemisphere'] = shape.index.to_series().apply(
#shape['Hemisphere'] = shape[subjName].apply(
#    lambda x: 'Left' if x.startswith('L') else 'Right' if x.startswith('flip-R') else None
#)
shape['Hemisphere'] = shape.index.to_series().astype(str).apply(
    lambda x: 'Left' if x.startswith('L') else 'Right' if x.startswith('flip-R') else None
)

shape = shape.reset_index()

#print(shape.head())
print(shape.columns)
print(shape)

Index(['subjName', 'iso1', 'iso2', 'iso3', 'iso4', 'iso5', 'iso6', 'UMAP1_U1',
       'UMAP1_U2', 'UMAP1_U3', 'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4', 'SubjID',
       'Study', 'Hemisphere'],
      dtype='object')
           subjName      iso1      iso2      iso3      iso4      iso5  \
0        L010017MHU -1.779720 -0.667942 -0.097973  1.451221 -0.281212   
1        L010023MHU -0.281341  0.077161  0.301623 -0.250930 -0.780885   
2        L010052MHU -0.784505 -0.755314  0.740993 -0.692597 -0.719726   
3        L010086MHU  0.095304  0.313703 -0.254081  0.338678 -0.089995   
4        L010172MHU  2.021033  3.362425 -0.214150  0.552886 -1.296802   
5        L010174MHU -0.392415 -0.039164 -0.350800 -0.017298 -0.587716   
6        L010179MHU  0.711163 -0.772441 -0.141060  0.604570 -1.184974   
7        L010210MHU -0.137848 -1.134967 -0.764875  0.481225 -0.086211   
8        L010218MHU -1.171972  1.688308 -0.410348 -1.080374  0.338789   
9        L010264MHU  1.183212 -0.933049 -1.122756 -0.782066 -

In [43]:
##################################### Prepare Database-specific infomation ####################################
# Load anatomical and functional info 
HDTRACK_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\{curProject}_INFO_result\HD_Track_cleaned_filled.csv'
WINHD_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\{curProject}_INFO_result\WIN_HD.csv'
MOTOHD_info_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\{curProject}_INFO_result\MOTO_HD_cleaned.csv'

HDTRACK_info, WINHD_info, MOTOHD_info = None, None, None

try:
    WINHD_info = pd.read_csv(WINHD_info_path, index_col=0, header=0, sep=';', encoding='latin1')
    MOTOHD_info = pd.read_csv(MOTOHD_info_path, index_col=0, header=0, sep=';', encoding='latin1')
    HDTRACK_info = pd.read_csv(HDTRACK_info_path, index_col=0, header=0, sep=';', encoding='latin1')
except FileNotFoundError as e:
    print(f"Error: {e}")

if WINHD_info is not None:  
    WINHD_info.index.name = 'SubjID'                     # rename index from the original 'File Name' to 'Subject'
    WINHD_info['SubjID'] = WINHD_info.index
    WINHD_info = WINHD_info.drop(columns=['Unnamed: 1'])  # remove col created by trailing ';' in CSV file
    WINHD_info.drop(['DDN','Date IRM','UHDRS'], axis=1, inplace=True)   
    WINHD_info = WINHD_info.rename(columns={"Age à l'IRM": "Age","Sexe":"Sex"})
    WINHD_info['Diag'] = WINHD_info['Diag'].replace('MH','MH_premanifest')
    WINHD_info['Study'] = 'WINHD'
if MOTOHD_info is not None:
    MOTOHD_info.index.name = 'SubjID'
    MOTOHD_info['SubjID'] = MOTOHD_info.index
    MOTOHD_info.drop(['DDN','Date IRM'], axis=1, inplace=True)
    MOTOHD_info = MOTOHD_info.rename(columns={"Age at IRM":"Age","Sexe":"Sex"})
    MOTOHD_info['Diag'] = np.where(MOTOHD_info['UHDRS'] >= 5, 'MH', 'MH_premanifest')  # create Diag col, set value <5 as MH_premanifest
    MOTOHD_info['Study'] = 'MOTOHD'
if HDTRACK_info is not None:
    HDTRACK_info.index.name = 'SubjID'
    HDTRACK_info['SubjID'] = HDTRACK_info.index
    HDTRACK_info = HDTRACK_info.drop(columns=['Unnamed: 1'])    # remove col created by trailing ';' in CSV file
    HDTRACK_info.drop(['DDN','Date IRM'],axis=1, inplace=True)
    HDTRACK_info = HDTRACK_info.rename(columns={"Age at IRM":"Age","Sexe":"Sex","Allèle muté":"CAG"," Allele normal":"CAG_normalAllele"})
    #df.loc[df['B'] < 5, 'Diag'] = 'aa'
    HDTRACK_info.loc[HDTRACK_info['UHDRS'] < 5,'Diag'] = 'MH_premanifest'
    # Not done: fill in normalAlle when absent with a standerd average
    #HDTRACK_info.loc[HDTRACK_info['Diag'] == 'T', ['CAG', 'CAG_normalAllele']] = \  # verified that all NA are controls, 18-20 repeats on average
    #HDTRACK_info.loc[HDTRACK_info['Diag'] == 'T', ['CAG', 'CAG_normalAllele']].fillna(18)
    HDTRACK_info['Study'] = 'HDTRACK'
else:
    print("Info could not be loaded. Aborting further steps.")
"""
print(len(WINHD_info))
print(WINHD_info.columns)
print(WINHD_info.head())
print("Original data types:\n", WINHD_info.dtypes)

print(len(MOTOHD_info))
print(MOTOHD_info.columns)
print(MOTOHD_info.head())
print("Original data types:\n", MOTOHD_info.dtypes)
"""
print(len(HDTRACK_info))
print(HDTRACK_info.columns)
print(HDTRACK_info.head())
print("Original data types:\n", HDTRACK_info.dtypes)

78
Index(['Sex', 'Age', 'Diag', 'CAG', 'CAG_normalAllele', 'UHDRS', 'SubjID',
       'Study'],
      dtype='object')
                 Sex  Age Diag   CAG  CAG_normalAllele  UHDRS  \
SubjID                                                          
v_000-517-510_S2   M   62   MH  42.0              18.0    NaN   
v_020-527-191_S2   F   64    T   NaN               NaN    NaN   
v_028-502-628_S2   F   44    T   NaN               NaN    NaN   
v_038-310-89X_S2   M   64   MH  40.0              10.0    NaN   
v_056-927-020_S2   F   38   MH  42.0              19.0    NaN   

                            SubjID    Study  
SubjID                                       
v_000-517-510_S2  v_000-517-510_S2  HDTRACK  
v_020-527-191_S2  v_020-527-191_S2  HDTRACK  
v_028-502-628_S2  v_028-502-628_S2  HDTRACK  
v_038-310-89X_S2  v_038-310-89X_S2  HDTRACK  
v_056-927-020_S2  v_056-927-020_S2  HDTRACK  
Original data types:
 Sex                  object
Age                   int64
Diag                 object

In [45]:
# Add additional pyramidal reflex info

MOTOHD_info_pyramidal = rf'{curRoot}:\B_projWIP\proj_{curProject}\{curProject}_INFO_result\INFO_files\Classsification-reflexes-MOTO HD.csv'
try:
    MOTOHD_info_pyramidal = pd.read_csv(MOTOHD_info_pyramidal, index_col=0, header=0, sep=';', encoding='latin1')
except FileNotFoundError as e:
    print(f"Error: {e}")
if MOTOHD_info_pyramidal is not None:
    MOTOHD_info_pyramidal.index.name = 'SubjID'

MOTOHD_combined_info = MOTOHD_info.join(MOTOHD_info_pyramidal)

print(MOTOHD_info)
print(MOTOHD_combined_info)

           Age  CAG  UHDRS Sex     SubjID            Diag   Study
SubjID                                                           
010052MHU   35   43      4   F  010052MHU  MH_premanifest  MOTOHD
011042MHU   37   43      4   F  011042MHU  MH_premanifest  MOTOHD
010697MHU   24   57     39   M  010697MHU              MH  MOTOHD
010290MHU   62   42     27   M  010290MHU              MH  MOTOHD
010606MHU   42   43      0   F  010606MHU  MH_premanifest  MOTOHD
010023MHU   63   42     14   M  010023MHU              MH  MOTOHD
011160MHU   44   42     23   F  011160MHU              MH  MOTOHD
010540MHU   61   39      4   F  010540MHU  MH_premanifest  MOTOHD
010179MHU   36   42      1   F  010179MHU  MH_premanifest  MOTOHD
010339MHU   30   41      1   F  010339MHU  MH_premanifest  MOTOHD
010017MHU   48   45      4   F  010017MHU  MH_premanifest  MOTOHD
011119MHU   66   40      4   M  011119MHU  MH_premanifest  MOTOHD
010820MHU   47   40      3   M  010820MHU  MH_premanifest  MOTOHD
010174MHU 

In [47]:
##################  Mering the three info files  ###################
info = pd.concat(
    [WINHD_info, MOTOHD_combined_info, HDTRACK_info],
    axis=0,      # stack rows
    ignore_index=True,   # reset row index
    sort=False   # keep only existing columns
)

print(info.head())

   Age            Diag Sex              SubjID  Study  CAG  UHDRS Pyramidal  \
0   38               T   M  WINHD_001_01_RO_S3  WINHD  NaN    NaN       NaN   
1   30  MH_premanifest   M  WINHD_001_02_OM_S3  WINHD  NaN    NaN       NaN   
2   35  MH_premanifest   M  WINHD_001_03_DA_S3  WINHD  NaN    NaN       NaN   
3   26  MH_premanifest   F  WINHD_001_04_DE_S3  WINHD  NaN    NaN       NaN   
4   38  MH_premanifest   M  WINHD_001_05_BS_S3  WINHD  NaN    NaN       NaN   

   CAG_normalAllele  
0               NaN  
1               NaN  
2               NaN  
3               NaN  
4               NaN  


In [87]:
##########################  Merging shape (Isomap/Umap) and merged_info  ###########################
merged_info = shape.merge(info, on=['SubjID','Study'], how='left')  # left join, keep all the rows
merged_info.set_index('subjName', inplace=True)
#print(merged_info.head())
print(merged_info.columns)
print(len(merged_info))

Index(['iso1', 'iso2', 'iso3', 'iso4', 'iso5', 'iso6', 'UMAP1_U1', 'UMAP1_U2',
       'UMAP1_U3', 'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4', 'SubjID', 'Study',
       'Hemisphere', 'Age', 'Diag', 'Sex', 'CAG', 'UHDRS', 'Pyramidal',
       'CAG_normalAllele'],
      dtype='object')
42


In [470]:
###########################################  Verifications  ###########################################

In [89]:
####################  Summary Group  ###################
study_counts = merged_info['Study'].value_counts()
print(study_counts)

####################  Get a summary of statistics  ####################
summary_stats = merged_info['iso1'].describe()
#print("\nSummary statistics for the 'isomap1' column:")
#print(summary_stats)

####################  Detect null values in a specific column  ###################
null_values = merged_info['Study'].isnull()
#print(null_values)

####################  Filter rows where the specified column has null values  #####################
null_rows = merged_info[merged_info['CAG_normalAllele'].isnull()]
#print(null_rows)
#print(null_rows.index)
print(null_rows.index.tolist())


Study
MOTOHD    42
Name: count, dtype: int64
['L010017MHU', 'L010023MHU', 'L010052MHU', 'L010086MHU', 'L010172MHU', 'L010174MHU', 'L010179MHU', 'L010210MHU', 'L010218MHU', 'L010264MHU', 'L010290MHU', 'L010339MHU', 'L010408MHU', 'L010540MHU', 'L010606MHU', 'L010697MHU', 'L010820MHU', 'L011006MHU', 'L011042MHU', 'L011119MHU', 'L011160MHU', 'flip-R010017MHU', 'flip-R010023MHU', 'flip-R010052MHU', 'flip-R010086MHU', 'flip-R010172MHU', 'flip-R010174MHU', 'flip-R010179MHU', 'flip-R010210MHU', 'flip-R010218MHU', 'flip-R010264MHU', 'flip-R010290MHU', 'flip-R010339MHU', 'flip-R010408MHU', 'flip-R010540MHU', 'flip-R010606MHU', 'flip-R010697MHU', 'flip-R010820MHU', 'flip-R011006MHU', 'flip-R011042MHU', 'flip-R011119MHU', 'flip-R011160MHU']


In [93]:
################################  Saving csv files if needed  #################################
file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_combined_k{curNeig}_{curDistType}.csv'
print(file_path)

# Write the DataFrame to a CSV file
#merged_info.to_csv(file_path, index=True)

################################  Test read the CSV file back into a DataFrame  ################################
#df_loaded = pd.read_csv(file_path)
#print(len(df_loaded))
#print("Data types:\n", df_loaded.dtypes)

C:\B_projWIP\proj_HD\Analysis_2025\CST_combined_k30_min.csv
